# Exact match or string matching measure (using fuzzy logic)

In [2]:
from google.colab import drive;
drive.mount('/content/drive');
!pip install pandas


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!pip install -U scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.4/13.4 MB 39.5 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.3.2
    Uninstalling scikit-learn-1.3.2:
      Successfully uninstalled scikit-learn-1.3.2


In [4]:
!pip install fuzzywuzzy pandas openpyxl

In [5]:
import pandas as pd

file_path = '/content/drive/MyDrive/Colab Notebooks/IST/Example/GPT_4_Turbo/ACAS_XU/Result/GPT_4_Turbo_Result.xlsx'
df = pd.read_excel(file_path)
print(df.head())

     Run                                           BlueROV2  \
0  Run 1  G1: Hazards for BlueROV2 addressed in accordan...   
1  Run 2  G1: Hazards for BlueROV2 are addressed in acco...   
2  Run 3  G1: Hazards for BlueROV2 addressed in accordan...   
3  Run 4  G1: Hazards for BlueROV2 system addressed in a...   
4  Run 5  G1: Hazards for BlueROV2 addressed in accordan...   

                                            DeepMind  \
0  G1: Interpretability Claim. The DeepMind ML Mo...   
1  G1: Interpretability Claim. The DeepMind diagn...   
2  G1: Interpretability Claim. The DeepMind ML mo...   
3  G1: Interpretability Claim. The DeepMind ML Mo...   
4  G1: Interpretability Claim. The DeepMind ML Mo...   

                                                GPCA  \
0  G1: Operational safety is verified in GPCA sys...   
1  G1: Operational safety is verified in GPCA sys...   
2  G1: Operational safety is verified in GPCA sys...   
3  G1: Operational safety is verified in GPCA sys...   
4  G

In [6]:
file_path2 = '/content/drive/MyDrive/Colab Notebooks/IST/Example/GPT_4_Turbo/ACAS_XU/Result/Ground_Truth_AC.xlsx'
ground_truth_df = pd.read_excel(file_path2)
print(ground_truth_df.head())

          Case                                                 AC
0     BlueROV2  G1: Hazards for BlueROV2 addressed in accordan...
1     DeepMind  G1: Interpretability Claim - The ML system is ...
2         GPCA  G1: Operational safety is verified in "GPCA sy...
3  IM_Software  G1: IM server software is acceptable secure in...


In [7]:
from fuzzywuzzy import fuzz


for index, experiment_row in df.iterrows():
    for case in ['BlueROV2', 'DeepMind', 'GPCA', 'IM_Software']:
        experiment_text = experiment_row[case]
        ground_truth_text = ground_truth_df.loc[ground_truth_df['Case'] == case, 'AC'].iloc[0]
        similarity_score = fuzz.ratio(experiment_text, ground_truth_text)/ 100.0

        print(f"Similarity score for {case} in {experiment_row['Run']}: {similarity_score}")
    print("-------")

Similarity score for BlueROV2 in Run 1: 0.75
Similarity score for DeepMind in Run 1: 0.27
Similarity score for GPCA in Run 1: 0.28
Similarity score for IM_Software in Run 1: 0.08
-------
Similarity score for BlueROV2 in Run 2: 0.78
Similarity score for DeepMind in Run 2: 0.23
Similarity score for GPCA in Run 2: 0.36
Similarity score for IM_Software in Run 2: 0.09
-------
Similarity score for BlueROV2 in Run 3: 0.78
Similarity score for DeepMind in Run 3: 0.23
Similarity score for GPCA in Run 3: 0.46
Similarity score for IM_Software in Run 3: 0.07
-------
Similarity score for BlueROV2 in Run 4: 0.79
Similarity score for DeepMind in Run 4: 0.25
Similarity score for GPCA in Run 4: 0.26
Similarity score for IM_Software in Run 4: 0.15
-------
Similarity score for BlueROV2 in Run 5: 0.74
Similarity score for DeepMind in Run 5: 0.26
Similarity score for GPCA in Run 5: 0.28
Similarity score for IM_Software in Run 5: 0.22
-------


/usr/local/lib/python3.10/dist-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


# Bleu score

In [8]:
!pip install sacrebleu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.0/104.0 kB 7.0 MB/s eta 0:00:00


In [9]:
import sacrebleu

def calculate_bleu_score(experiment_text, ground_truth_text):
    references = [[ground_truth_text]]
    system_output = [experiment_text]
    bleu = sacrebleu.corpus_bleu(system_output, references)
    # Scale the score to a range between 0 and 1 and round to two decimal points
    return round(bleu.score / 100, 2)

for case in ['BlueROV2', 'DeepMind', 'GPCA', 'IM_Software']:
    print(f"Calculating BLEU scores for {case}:")
    for index, row in df.iterrows():
        experiment_text = row[case]
        ground_truth_text = ground_truth_df.loc[ground_truth_df['Case'] == case, 'AC'].iloc[0]

        if pd.notnull(experiment_text) and pd.notnull(ground_truth_text):
            bleu_score = calculate_bleu_score(experiment_text, ground_truth_text)
            print(f"Run {index + 1} BLEU score: {bleu_score}")
        else:
            print(f"Run {index + 1} BLEU score: Data Missing")
    print("-------")


Calculating BLEU scores for BlueROV2:
Run 1 BLEU score: 0.6
Run 2 BLEU score: 0.61
Run 3 BLEU score: 0.59
Run 4 BLEU score: 0.63
Run 5 BLEU score: 0.55
-------
Calculating BLEU scores for DeepMind:
Run 1 BLEU score: 0.18
Run 2 BLEU score: 0.16
Run 3 BLEU score: 0.16
Run 4 BLEU score: 0.16
Run 5 BLEU score: 0.19
-------
Calculating BLEU scores for GPCA:
Run 1 BLEU score: 0.18
Run 2 BLEU score: 0.26
Run 3 BLEU score: 0.24
Run 4 BLEU score: 0.22
Run 5 BLEU score: 0.25
-------
Calculating BLEU scores for IM_Software:
Run 1 BLEU score: 0.25
Run 2 BLEU score: 0.19
Run 3 BLEU score: 0.14
Run 4 BLEU score: 0.23
Run 5 BLEU score: 0.14
-------


# Semantic Similarity

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer()

for case in ['BlueROV2', 'DeepMind', 'GPCA', 'IM_Software']:
    experiment_texts = df[case].tolist()
    ground_truth_text = ground_truth_df.loc[ground_truth_df['Case'] == case, 'AC'].iloc[0]

    combined_texts = experiment_texts + [ground_truth_text]

    tfidf_matrix = vectorizer.fit_transform(combined_texts)

    experiment_vectors = tfidf_matrix[:-1]
    ground_truth_vector = tfidf_matrix[-1:]

    cosine_similarities = cosine_similarity(experiment_vectors, ground_truth_vector)

    for i, similarity_score in enumerate(cosine_similarities.flatten(), 1):
        # Round the score to two decimal places without multiplying by 100
        rounded_score = round(similarity_score, 2)
        print(f"Cosine similarity for {case}, Run {i}: {rounded_score}")
    print("-------")


Cosine similarity for BlueROV2, Run 1: 0.91
Cosine similarity for BlueROV2, Run 2: 0.83
Cosine similarity for BlueROV2, Run 3: 0.88
Cosine similarity for BlueROV2, Run 4: 0.86
Cosine similarity for BlueROV2, Run 5: 0.85
-------
Cosine similarity for DeepMind, Run 1: 0.52
Cosine similarity for DeepMind, Run 2: 0.6
Cosine similarity for DeepMind, Run 3: 0.54
Cosine similarity for DeepMind, Run 4: 0.55
Cosine similarity for DeepMind, Run 5: 0.53
-------
Cosine similarity for GPCA, Run 1: 0.51
Cosine similarity for GPCA, Run 2: 0.55
Cosine similarity for GPCA, Run 3: 0.4
Cosine similarity for GPCA, Run 4: 0.59
Cosine similarity for GPCA, Run 5: 0.62
-------
Cosine similarity for IM_Software, Run 1: 0.7
Cosine similarity for IM_Software, Run 2: 0.65
Cosine similarity for IM_Software, Run 3: 0.61
Cosine similarity for IM_Software, Run 4: 0.7
Cosine similarity for IM_Software, Run 5: 0.63
-------
